In [ ]:
import re
import string
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from pathlib import Path
from scipy.sparse import hstack, csr_matrix

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report,
)

pd.set_option("display.max_columns", None)
plt.style.use("ggplot")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
import os
os.makedirs("models", exist_ok=True)
os.makedirs("reports/figures", exist_ok=True)


In [ ]:
def find_file(filename):
    candidates = [
        Path(filename),
        Path("data/processed") / filename,
        Path("../data/processed") / filename,
        Path("../../data/processed") / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find '{filename}'. Looked in: {[str(c) for c in candidates]}\n"
        f"Fix: copy '{filename}' into the same folder as this notebook, "
        f"or into 'data/processed/' relative to the project root."
    )

train_df = pd.read_csv(find_file("train_features.csv"))
test_df = pd.read_csv(find_file("test_features.csv"))

train_df["name"] = train_df["name"].fillna("")
test_df["name"] = test_df["name"].fillna("")

print("train_df:", train_df.shape, " test_df:", test_df.shape)
train_df[["name", "target"]].head()


## Step 1 — Clean the campaign title text


In [ ]:
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)   # drop punctuation/digits
    text = re.sub(r"\s+", " ", text).strip()  # collapse whitespace
    return text

train_df["name_clean"] = train_df["name"].apply(clean_text)
test_df["name_clean"] = test_df["name"].apply(clean_text)

train_df[["name", "name_clean"]].head(10)


## Step 2 — TF-IDF vectorization


In [ ]:
tfidf = TfidfVectorizer(
    max_features=2500,
    stop_words="english",
    ngram_range=(1, 1),
    min_df=2,
)

X_train_tfidf = tfidf.fit_transform(train_df["name_clean"])
X_test_tfidf = tfidf.transform(test_df["name_clean"])

print("TF-IDF train matrix:", X_train_tfidf.shape)
print("TF-IDF test matrix: ", X_test_tfidf.shape)
print("Sample vocabulary:", tfidf.get_feature_names_out()[:20])

## Step 3 — Prepare the tabular matrix


In [ ]:
categorical_cols = ["main_category", "currency", "country", "launch_weekday"]
numeric_cols = [
    "usd_goal_real", "goal_log", "campaign_duration",
    "launch_month", "launch_quarter",
    "category_frequency", "country_success_rate",
]
tabular_cols = categorical_cols + numeric_cols

tabular_preprocessor = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ("num", StandardScaler(), numeric_cols),
])

X_train_tabular = tabular_preprocessor.fit_transform(train_df[tabular_cols])
X_test_tabular = tabular_preprocessor.transform(test_df[tabular_cols])

X_train_tabular = csr_matrix(X_train_tabular)
X_test_tabular = csr_matrix(X_test_tabular)

tabular_feature_names = tabular_preprocessor.get_feature_names_out()
print("Tabular train matrix:", X_train_tabular.shape)


## Step 4 — Fuse text + tabular via `scipy.sparse.hstack`


In [ ]:
X_train_fused = hstack([X_train_tfidf, X_train_tabular]).tocsr()
X_test_fused = hstack([X_test_tfidf, X_test_tabular]).tocsr()

y_train = train_df["target"]
y_test = test_df["target"]

fused_feature_names = np.concatenate([
    tfidf.get_feature_names_out(),
    tabular_feature_names,
])

print("Fused train matrix:", X_train_fused.shape)
print("Fused test matrix: ", X_test_fused.shape)
print(f"({X_train_tfidf.shape[1]} text features + {X_train_tabular.shape[1]} tabular features)")


## Step 5 — Train a regularized Logistic Regression on the fused matrix


In [ ]:
fused_model = LogisticRegression(
    penalty="l2",
    C=1.0,
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
fused_model.fit(X_train_fused, y_train)

y_pred = fused_model.predict(X_test_fused)
y_proba = fused_model.predict_proba(X_test_fused)[:, 1]

fused_results = {
    "Model": "Logistic Regression (TF-IDF + tabular fusion)",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_proba),
}
pd.DataFrame([fused_results]).round(4)


### Compare against a tabular-only baseline (same model, no text)


In [ ]:
tabular_only_model = LogisticRegression(
    penalty="l2", C=1.0, max_iter=1000,
    class_weight="balanced", random_state=RANDOM_STATE,
)
tabular_only_model.fit(X_train_tabular, y_train)

y_pred_tab = tabular_only_model.predict(X_test_tabular)
y_proba_tab = tabular_only_model.predict_proba(X_test_tabular)[:, 1]

tabular_only_results = {
    "Model": "Logistic Regression (tabular only)",
    "Accuracy": accuracy_score(y_test, y_pred_tab),
    "Precision": precision_score(y_test, y_pred_tab),
    "Recall": recall_score(y_test, y_pred_tab),
    "F1": f1_score(y_test, y_pred_tab),
    "ROC-AUC": roc_auc_score(y_test, y_proba_tab),
}

comparison_df = pd.DataFrame([tabular_only_results, fused_results]).round(4)
comparison_df.to_csv("reports/tfidf_fusion_comparison.csv", index=False)
comparison_df


## Step 6 — Feature weight analysis


In [ ]:
coefs = fused_model.coef_[0]
n_text_features = X_train_tfidf.shape[1]

text_coefs = pd.DataFrame({
    "token": fused_feature_names[:n_text_features],
    "coefficient": coefs[:n_text_features],
})
tabular_coefs = pd.DataFrame({
    "feature": fused_feature_names[n_text_features:],
    "coefficient": coefs[n_text_features:],
})

top_positive_tokens = text_coefs.sort_values("coefficient", ascending=False).head(15)
top_negative_tokens = text_coefs.sort_values("coefficient").head(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.barplot(data=top_positive_tokens, x="coefficient", y="token", ax=axes[0], color="seagreen")
axes[0].set_title("Top 15 Tokens → Success")
sns.barplot(data=top_negative_tokens, x="coefficient", y="token", ax=axes[1], color="indianred")
axes[1].set_title("Top 15 Tokens → Failure")
plt.tight_layout()
plt.savefig("reports/figures/tfidf_top_tokens.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
top_tabular = tabular_coefs.reindex(tabular_coefs.coefficient.abs().sort_values(ascending=False).index).head(15)

plt.figure(figsize=(10, 7))
sns.barplot(data=top_tabular, x="coefficient", y="feature",
            palette=["seagreen" if c > 0 else "indianred" for c in top_tabular["coefficient"]])
plt.title("Top 15 Tabular Feature Coefficients")
plt.tight_layout()
plt.savefig("reports/figures/tfidf_tabular_coefficients.png", dpi=300, bbox_inches="tight")
plt.show()


### Save deliverables

In [ ]:
text_coefs.sort_values("coefficient", ascending=False).to_csv(
    "reports/tfidf_text_token_coefficients.csv", index=False
)
tabular_coefs.to_csv("reports/tfidf_tabular_coefficients.csv", index=False)

joblib.dump({
    "tfidf_vectorizer": tfidf,
    "tabular_preprocessor": tabular_preprocessor,
    "model": fused_model,
}, "models/tfidf_fusion_model.pkl")

print("Saved: models/tfidf_fusion_model.pkl")
print("Saved: reports/tfidf_text_token_coefficients.csv")
print("Saved: reports/tfidf_tabular_coefficients.csv")

import shutil
shutil.make_archive("tfidf_fusion_outputs", "zip", "reports")

try:
    from google.colab import files
    files.download("tfidf_fusion_outputs.zip")
except ImportError:
    print("Not running in Colab -- saved locally: tfidf_fusion_outputs.zip")
